# Segment Wise Modelling

This notebook runs segment-wise **one-week-ahead** tea price forecasting on `tea_preprocessed_v2.csv` for four segments: High Grown, Low Grown, Off-Grade, and Dust.

It creates a next-auction target and evaluates XGBoost, LightGBM, Gradient Boosting, and Random Forest with mandatory time-series cross-validation to avoid data leakage.

Features include engineered columns from `feature_engineering.py`, lagged weather variables (1/2/3-week lags for precipitation, sunshine, and temperature), and structural features (`grade_rank`, `tier_rank`, `prestige_rank`).

In [ ]:
"""Imports and global config for segment-wise time-series forecasting."""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Please install lightgbm: pip install lightgbm") from exc

print("Libraries loaded.")

Libraries loaded.


In [ ]:
"""Load data, build one-week-ahead target, validate engineered input, and prepare segment dataframes."""

df_raw = pd.read_csv("../data/Processed/tea_preprocessed_v2.csv")
TARGET_BASE = "price_mid_lkr"
TARGET = "price_mid_lkr_t_plus_1w"
FORECAST_HORIZON = "t_plus_1_week"
SEGMENT_COL = "category_type"

# Keep the original row count for traceability to source dataset size.
raw_rows = len(df_raw)
df = df_raw.copy()

# Basic cleaning for robust model training.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_BASE, SEGMENT_COL]).copy()

# Stable chronological ordering before creating t+1 target.
if "sale_year" in df.columns:
    df["_sale_year_num"] = pd.to_numeric(df["sale_year"], errors="coerce")
else:
    df["_sale_year_num"] = np.nan
if "sale_number" in df.columns:
    df["_sale_num"] = pd.to_numeric(df["sale_number"], errors="coerce")
else:
    df["_sale_num"] = np.nan
df["_sale_id_norm"] = pd.to_numeric(df.get("sale_id", np.nan), errors="coerce")

df = df.sort_values(["_sale_year_num", "_sale_num", "_sale_id_norm"], kind="mergesort").reset_index(drop=True)

# One-week-ahead forecast target within each segment stream.
df[TARGET] = df.groupby(SEGMENT_COL, sort=False)[TARGET_BASE].shift(-1)
df = df.dropna(subset=[TARGET]).copy()
df = df.drop(columns=["_sale_year_num", "_sale_num", "_sale_id_norm"] )

# Validate that feature_engineering.py outputs are present in this input.
fe_interact = [c for c in df.columns if c.startswith("interact__")]
fe_roll = [c for c in df.columns if c.startswith("roll3_")]
fe_poly = [c for c in df.columns if c.startswith("poly2__")]
if not (fe_interact and fe_roll and fe_poly):
    raise ValueError(
        "Feature-engineered columns not found. Please generate tea_preprocessed_v2.csv using feature_engineering.py."
    )

# Structural features requested in the methodology.
grade_rank_map = {
    "bop": 4, "bopf": 4, "bop1": 4,
    "fbop": 3, "fbop1": 3, "fbopf": 3, "fbopf1": 3, "fbopf_tippy": 3,
    "op": 2, "op1": 2, "opa": 2,
    "pekoe": 1, "pek1": 1, "pekoe_fbop": 1,
}
tier_rank_map = {"select_best": 4, "best": 3, "below_best": 2, "others": 1}
prestige_rank_map = {"high": 3, "medium": 2, "low": 1}

df["grade_rank"] = df.get("grade", pd.Series("", index=df.index)).astype(str).str.lower().map(grade_rank_map).fillna(0)
df["tier_rank"] = df.get("tier", pd.Series("", index=df.index)).astype(str).str.lower().map(tier_rank_map).fillna(df.get("tier_enc", 0)).fillna(0)
df["prestige_rank"] = df.get("elevation", pd.Series("", index=df.index)).astype(str).str.lower().map(prestige_rank_map).fillna(0)

# Required lagged weather variables: lag1/lag2/lag3 for precipitation, sunshine, temperature.
lag_weather_cols = [
    c for c in df.columns
    if ("lag1" in c or "lag2" in c or "lag3" in c)
    and ("precipitation" in c or "sunshine" in c or "temperature" in c)
]

# Remove target-derived and same-auction price proxy columns to prevent leakage.
drop_cols = {
    TARGET_BASE,
    TARGET,
    "price_mid_lkr_log",
    "price_mid_usd",
    "price_lo_lkr",
    "price_hi_lkr",
    "price_range_lkr",
    "sale_id",
    "sale_date_raw",
    "has_price_target",
}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_cols if c not in drop_cols]

# Force required methodological features into feature set if present.
required_structural = ["grade_rank", "tier_rank", "prestige_rank"]
required_features = [c for c in (lag_weather_cols + required_structural) if c in df.columns]
feature_cols = sorted(list(set(candidate_features + required_features)))

# Create 4 required segment dataframes.
segments = {
    "high_grown": df[df[SEGMENT_COL] == "high_grown"].copy(),
    "low_grown": df[df[SEGMENT_COL] == "low_grown"].copy(),
    "off_grade": df[df[SEGMENT_COL] == "off_grade"].copy(),
    "dust": df[df[SEGMENT_COL] == "dust"].copy(),
}

segment_labels = {
    "high_grown": "High Grown",
    "low_grown": "Low Grown",
    "off_grade": "Off-Grade",
    "dust": "Dust",
}

print(f"Rows loaded from source: {raw_rows}")
print(f"Rows used after forecast-target construction: {len(df)}")
print(f"Forecast horizon: {FORECAST_HORIZON}")
print(f"Engineered feature counts -> interact: {len(fe_interact)}, roll3: {len(fe_roll)}, poly2: {len(fe_poly)}")
print(f"Lag-weather features found: {len(lag_weather_cols)}")
print(f"Total modeling features (leakage-safe): {len(feature_cols)}")
for key, sdf in segments.items():
    print(f"{segment_labels[key]:<10s}: {len(sdf)} rows")

Rows loaded from source: 3034
Rows used after target cleanup: 2886
Engineered feature counts -> interact: 28, roll3: 19, poly2: 15
Lag-weather features found: 36
Total modeling features (leakage-safe): 231
High Grown: 516 rows
Low Grown : 1348 rows
Off-Grade : 499 rows
Dust      : 523 rows


In [10]:
"""Define models and adaptive time-series CV evaluation per segment."""

models = {
    "XGBoost": XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

BASE_CV_SPLITS = 5
MIN_CV_SPLITS = 2
MIN_TEST_ROWS_PER_FOLD = 30
MIN_INITIAL_TRAIN_ROWS = 60

def choose_adaptive_splits(n_rows: int, base_splits: int = BASE_CV_SPLITS) -> int | None:
    """Choose fewer CV splits for small segments to keep train/test folds meaningful."""
    max_candidate = min(base_splits, n_rows - 1)
    for n_splits in range(max_candidate, MIN_CV_SPLITS - 1, -1):
        # For default TimeSeriesSplit: test_size ~= n_rows // (n_splits + 1)
        test_size = n_rows // (n_splits + 1)
        initial_train_size = n_rows - (n_splits * test_size)
        if test_size >= MIN_TEST_ROWS_PER_FOLD and initial_train_size >= MIN_INITIAL_TRAIN_ROWS:
            return n_splits

    # Fallback minimum only if technically feasible.
    if n_rows > MIN_CV_SPLITS:
        return MIN_CV_SPLITS
    return None

def compute_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R2 on raw LKR prices."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

def evaluate_segment(segment_df, segment_name):
    """Run adaptive-fold TimeSeriesSplit for all models on one segment."""
    # Sort by auction order so each fold trains on past and tests on future only.
    sort_cols = [c for c in ["sale_year", "sale_number"] if c in segment_df.columns]
    if sort_cols:
        segment_df = segment_df.sort_values(sort_cols).reset_index(drop=True)

    X_seg = segment_df[feature_cols].copy()
    y_seg = segment_df[TARGET].copy()

    n_splits = choose_adaptive_splits(len(segment_df))
    if n_splits is None:
        raise ValueError(f"Not enough rows for time-series CV in segment {segment_name}: {len(segment_df)}")

    # Fold-safe imputation happens inside each fold through a pipeline.
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seg), start=1):
        X_train, X_test = X_seg.iloc[train_idx], X_seg.iloc[test_idx]
        y_train, y_test = y_seg.iloc[train_idx], y_seg.iloc[test_idx]

        for model_name, model in models.items():
            pipe = Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("model", clone(model)),
            ])
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)
            rmse, mae, mape, r2 = compute_metrics(y_test, preds)

            fold_rows.append(
                {
                    "Segment": segment_name,
                    "Fold": fold_idx,
                    "Model": model_name,
                    "RMSE": rmse,
                    "MAE": mae,
                    "MAPE": mape,
                    "R2": r2,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "n_splits_used": n_splits,
                }
            )

    fold_results = pd.DataFrame(fold_rows)
    summary = (
        fold_results.groupby(["Segment", "Model"], as_index=False)
        .agg(
            RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
            MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
            MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
            R2_mean=("R2", "mean"), R2_std=("R2", "std"),
            CV_Splits_Used=("n_splits_used", "max"),
        )
        .sort_values(["Segment", "RMSE_mean"])
        .reset_index(drop=True)
    )
    return fold_results, summary

In [ ]:
"""Run segment-wise CV for all segments, show results, and save best-per-segment forecast table."""

all_fold_results = []
all_summary_results = []

for seg_key in ["high_grown", "low_grown", "off_grade", "dust"]:
    seg_df = segments[seg_key]
    seg_name = segment_labels[seg_key]

    if len(seg_df) < 30:
        print(f"Skipping {seg_name}: not enough rows for stable 5-fold CV.")
        continue

    fold_df, summary_df = evaluate_segment(seg_df, seg_name)
    all_fold_results.append(fold_df)
    all_summary_results.append(summary_df)
    print(f"Completed: {seg_name}")

cv_fold_results = pd.concat(all_fold_results, ignore_index=True)
cv_summary_results = pd.concat(all_summary_results, ignore_index=True)
cv_fold_results["Forecast_Horizon"] = FORECAST_HORIZON
cv_summary_results["Forecast_Horizon"] = FORECAST_HORIZON

print("\nSegment-wise 5-fold TimeSeries CV summary (sorted by RMSE):")
display(cv_summary_results.round(4))

print("\nBest model per segment (lowest RMSE_mean):")
best_per_segment = (
    cv_summary_results.sort_values(["Segment", "RMSE_mean"])
.groupby("Segment", as_index=False)
.first()[["Segment", "Model", "RMSE_mean", "MAE_mean", "MAPE_mean", "R2_mean", "Forecast_Horizon"]]
.sort_values("Segment")
.reset_index(drop=True)
.round(4)
)
display(best_per_segment)

tables_dir = Path("../reports/tables")
tables_dir.mkdir(parents=True, exist_ok=True)
best_path = tables_dir / "segmentwise_best_model_per_segment.csv"
best_per_segment.to_csv(best_path, index=False)
print(f"Saved: {best_path}")

Completed: High Grown
Completed: Low Grown
Completed: Off-Grade
Completed: Dust

Segment-wise 5-fold TimeSeries CV summary (sorted by RMSE):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,CV_Splits_Used
0,High Grown,Random Forest,224.7682,28.2667,156.6436,23.2764,18.9225,3.2855,0.2871,0.0860,5
1,High Grown,LightGBM,229.9959,34.0834,165.4046,26.3468,19.8451,3.7087,0.2588,0.0580,5
2,High Grown,XGBoost,234.0763,34.7721,162.9035,31.5620,19.8750,4.1395,0.2313,0.0838,5
3,High Grown,Gradient Boosting,238.4481,29.0502,168.1638,29.0486,20.1998,3.9113,0.1947,0.1222,5
4,Low Grown,Gradient Boosting,280.5365,62.1155,153.7782,19.8129,8.9725,1.0304,0.8581,0.0654,5
5,Low Grown,Random Forest,285.7480,81.3579,155.7914,32.4983,9.0175,1.6040,0.8469,0.0913,5
6,Low Grown,LightGBM,344.0631,135.2432,183.3618,56.0898,10.7604,2.6002,0.7607,0.1795,5
7,Low Grown,XGBoost,354.6926,105.0049,173.2753,36.6366,9.2957,0.9450,0.7649,0.1256,5
8,Off-Grade,XGBoost,145.1082,25.1306,106.9364,15.9521,15.4117,3.4502,0.2832,0.2275,5
9,Off-Grade,Random Forest,145.7976,26.7423,109.4378,16.7224,15.7240,3.8444,0.2737,0.2549,5



Best model per segment (lowest RMSE_mean):


,Segment,Model,RMSE_mean,MAE_mean,MAPE_mean,R2_mean
0,Dust,Random Forest,157.9989,109.9174,12.9115,0.4151
1,High Grown,Random Forest,224.7682,156.6436,18.9225,0.2871
2,Low Grown,Gradient Boosting,280.5365,153.7782,8.9725,0.8581
3,Off-Grade,XGBoost,145.1082,106.9364,15.4117,0.2832


## Hyperparameter Tuning

This section applies the following hyperparameter tuning pipeline:
- 5-fold TimeSeriesSplit
- GridSearchCV
- tuning is done separately for each segment and for each model

In [6]:
"""Run segment-wise GridSearchCV for all models (no best-model filtering)."""

import json
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone

# Provide defaults in-cell so tuning runs even if helper defs were not executed earlier.
if "PARAM_GRIDS_ALL_MODELS" not in globals():
    PARAM_GRIDS_ALL_MODELS = {
        "Random Forest": {
            "model__n_estimators": [200, 300],
            "model__max_depth": [None, 10],
            "model__min_samples_split": [2, 5],
        },
        "Gradient Boosting": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [2, 3],
        },
        "XGBoost": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [4, 6],
        },
        "LightGBM": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__num_leaves": [31, 63],
        },
    }

def build_tuning_estimator(model_name, base_model):
    """Create leakage-safe estimator for GridSearchCV."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", clone(base_model)),
    ])

def evaluate_tuned_estimator_timeseries(sdf, estimator, n_splits=5):
    """Evaluate tuned estimator with chronological CV on one segment."""
    sort_cols = [c for c in ["sale_year", "sale_number"] if c in sdf.columns]
    if sort_cols:
        sdf = sdf.sort_values(sort_cols).reset_index(drop=True)

    X_seg = sdf[feature_cols].copy()
    y_seg = sdf[TARGET].copy()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seg), start=1):
        X_train, X_test = X_seg.iloc[train_idx], X_seg.iloc[test_idx]
        y_train, y_test = y_seg.iloc[train_idx], y_seg.iloc[test_idx]

        est = clone(estimator)
        est.fit(X_train, y_train)
        preds = est.predict(X_test)
        rmse, mae, mape, r2 = compute_metrics(y_test, preds)

        rows.append({
            "Fold": fold_idx,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2,
        })
    return pd.DataFrame(rows)

inner_cv = TimeSeriesSplit(n_splits=5)

all_tuning_logs = []
all_tuned_metrics = []

for seg_name, sdf in segments.items():
    print(f"\nTuning segment: {seg_name} | rows={len(sdf)}")

    if len(sdf) < 20:
        print("  Skipped: not enough rows for stable 5-fold tuning.")
        continue

    X_seg = sdf[feature_cols].copy()
    y_seg = sdf[TARGET].copy()

    for model_name, base_model in models.items():
        if model_name not in PARAM_GRIDS_ALL_MODELS:
            print(f"  Skipped {model_name}: no parameter grid found.")
            continue

        estimator = build_tuning_estimator(model_name, base_model)
        param_grid = PARAM_GRIDS_ALL_MODELS[model_name]

        gscv = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            return_train_score=False,
        )
        gscv.fit(X_seg, y_seg)

        # Keep full grid search logs for each segment and model.
        grid_log = pd.DataFrame(gscv.cv_results_)[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].copy()
        grid_log["Segment"] = seg_name
        grid_log["Model"] = model_name
        grid_log["mean_test_RMSE"] = -grid_log["mean_test_score"]
        all_tuning_logs.append(grid_log)

        # Evaluate tuned estimator again with 5-fold time-series splits.
        tuned_fold_eval = evaluate_tuned_estimator_timeseries(
            sdf=sdf,
            estimator=gscv.best_estimator_,
            n_splits=5,
        )

        all_tuned_metrics.append({
            "Segment": seg_name,
            "Model": model_name,
            "RMSE_mean": tuned_fold_eval["RMSE"].mean(),
            "RMSE_std": tuned_fold_eval["RMSE"].std(),
            "MAE_mean": tuned_fold_eval["MAE"].mean(),
            "MAE_std": tuned_fold_eval["MAE"].std(),
            "MAPE_mean": tuned_fold_eval["MAPE"].mean(),
            "MAPE_std": tuned_fold_eval["MAPE"].std(),
            "R2_mean": tuned_fold_eval["R2"].mean(),
            "R2_std": tuned_fold_eval["R2"].std(),
            "Selected_Hyperparameters": json.dumps(gscv.best_params_, sort_keys=True),
        })

        print(f"  Done: {model_name} | tuned RMSE={tuned_fold_eval['RMSE'].mean():.2f}")

segmentwise_tuning_logs = pd.concat(all_tuning_logs, ignore_index=True) if all_tuning_logs else pd.DataFrame()
segmentwise_tuned_summary = pd.DataFrame(all_tuned_metrics)

print("\nTuning logs shape:", segmentwise_tuning_logs.shape)
print("Tuned summary shape:", segmentwise_tuned_summary.shape)

display(
    segmentwise_tuned_summary
    .sort_values(["Segment", "RMSE_mean"])
    .reset_index(drop=True)
    .round(4)
)


Tuning segment: high_grown | rows=516
  Done: XGBoost | tuned RMSE=233.50
  Done: LightGBM | tuned RMSE=227.74
  Done: Gradient Boosting | tuned RMSE=232.11
  Done: Random Forest | tuned RMSE=225.33

Tuning segment: low_grown | rows=1348
  Done: XGBoost | tuned RMSE=354.49
  Done: LightGBM | tuned RMSE=341.09
  Done: Gradient Boosting | tuned RMSE=278.66
  Done: Random Forest | tuned RMSE=285.75

Tuning segment: off_grade | rows=499
  Done: XGBoost | tuned RMSE=145.72
  Done: LightGBM | tuned RMSE=150.42
  Done: Gradient Boosting | tuned RMSE=161.32
  Done: Random Forest | tuned RMSE=145.80

Tuning segment: dust | rows=523
  Done: XGBoost | tuned RMSE=158.44
  Done: LightGBM | tuned RMSE=161.14
  Done: Gradient Boosting | tuned RMSE=170.17
  Done: Random Forest | tuned RMSE=158.00

Tuning logs shape: (128, 7)
Tuned summary shape: (16, 11)


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters
0,dust,Random Forest,157.9989,14.6945,109.9174,10.6564,12.9115,1.4111,0.4151,0.1533,"{""model__max_depth"": null, ""model__min_samples..."
1,dust,XGBoost,158.4356,23.7198,111.5719,14.8468,13.2380,2.1119,0.4043,0.2135,"{""model__learning_rate"": 0.05, ""model__max_dep..."
2,dust,LightGBM,161.1379,20.8432,115.5919,16.5197,13.6848,2.1962,0.3913,0.1763,"{""model__learning_rate"": 0.03, ""model__n_estim..."
3,dust,Gradient Boosting,170.1659,29.1139,120.5475,17.4954,14.2160,2.1290,0.2981,0.3346,"{""model__learning_rate"": 0.03, ""model__max_dep..."
4,high_grown,Random Forest,225.3322,28.7180,156.9024,23.1968,19.0154,3.3432,0.2839,0.0847,"{""model__max_depth"": 10, ""model__min_samples_s..."
5,high_grown,LightGBM,227.7371,33.3049,161.4617,25.9648,19.6037,3.7562,0.2737,0.0459,"{""model__learning_rate"": 0.03, ""model__n_estim..."
6,high_grown,Gradient Boosting,232.1093,33.0956,162.8220,29.7617,19.8924,4.4911,0.2442,0.0692,"{""model__learning_rate"": 0.03, ""model__max_dep..."
7,high_grown,XGBoost,233.4988,31.8901,161.4631,26.6298,19.7608,3.8562,0.2340,0.0721,"{""model__learning_rate"": 0.03, ""model__max_dep..."
8,low_grown,Gradient Boosting,278.6583,63.6215,151.6967,19.9301,8.8090,0.9629,0.8599,0.0661,"{""model__learning_rate"": 0.05, ""model__max_dep..."
9,low_grown,Random Forest,285.7480,81.3579,155.7914,32.4983,9.0175,1.6040,0.8469,0.0913,"{""model__max_depth"": null, ""model__min_samples..."


In [12]:
import os
import numpy as np
import pandas as pd

# Allow running this cell directly by loading prior outputs from disk when needed.
summary_csv = "../reports/tables/segmentwise_hyperparam_summary_all_models.csv"
logs_csv = "../reports/tables/_intermediate/segmentwise_hyperparam_gridsearch_all_results.csv"

if "segmentwise_tuned_summary" not in globals() or segmentwise_tuned_summary is None or segmentwise_tuned_summary.empty:
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(
            f"Missing {summary_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuned_summary = pd.read_csv(summary_csv)
    print(f"Loaded segmentwise_tuned_summary from: {summary_csv}")

if "segmentwise_tuning_logs" not in globals() or segmentwise_tuning_logs is None or segmentwise_tuning_logs.empty:
    if not os.path.exists(logs_csv):
        raise FileNotFoundError(
            f"Missing {logs_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuning_logs = pd.read_csv(logs_csv)
    print(f"Loaded segmentwise_tuning_logs from: {logs_csv}")

# 1) Detailed model comparison table (all segment-model rows).
performance_table = segmentwise_tuned_summary.copy()
performance_table["RMSE_rank_in_segment"] = performance_table.groupby("Segment")["RMSE_mean"].rank(method="dense", ascending=True)
performance_table["R2_rank_in_segment"] = performance_table.groupby("Segment")["R2_mean"].rank(method="dense", ascending=False)
performance_table["RMSE_cv_pct"] = (performance_table["RMSE_std"] / performance_table["RMSE_mean"]) * 100.0
performance_table["R2_cv_pct"] = (performance_table["R2_std"].abs() / performance_table["R2_mean"].abs().replace(0, np.nan)) * 100.0

# 2) Grid-search breadth table (shows how much parameter space was explored for each model in each segment).
grid_coverage = (
    segmentwise_tuning_logs
    .groupby(["Segment", "Model"], as_index=False)
    .agg(
        Param_Combinations_Tried=("params", "count"),
        Grid_Best_CV_RMSE=("mean_test_RMSE", "min"),
        Grid_Worst_CV_RMSE=("mean_test_RMSE", "max"),
        Grid_CV_RMSE_STD_Avg=("std_test_score", "mean"),
    )
)

# 3) Final findings table: performance + tuning breadth in one place.
all_findings_table = (
    performance_table
    .merge(grid_coverage, on=["Segment", "Model"], how="left")
    .sort_values(["Segment", "RMSE_mean", "R2_mean"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# 4) Reader-friendly combined snapshot for quick comparison.
segment_model_performance = all_findings_table[[
    "Segment",
    "Model",
    "RMSE_mean",
    "R2_mean",
    "RMSE_rank_in_segment",
    "R2_rank_in_segment",
    "RMSE_cv_pct",
    "R2_cv_pct",
    "Param_Combinations_Tried",
    "Grid_Best_CV_RMSE",
    "Grid_Worst_CV_RMSE",
    "Grid_CV_RMSE_STD_Avg",
]].copy()

print("All Findings Table (one row per segment and model):")
display(all_findings_table.round(4))

print("\nCombined RMSE/R2 performance table:")
display(segment_model_performance.round(4))

# Optional export so the same tables can go into report writing.
all_findings_table.to_csv("../reports/tables/segmentwise_all_findings_table.csv", index=False)
segment_model_performance.to_csv("../reports/tables/segmentwise_rmse_r2_combined.csv", index=False)

print("\nSaved:")
print("- ../reports/tables/segmentwise_all_findings_table.csv")
print("- ../reports/tables/segmentwise_rmse_r2_combined.csv")

All Findings Table (one row per segment and model):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters,RMSE_rank_in_segment,R2_rank_in_segment,RMSE_cv_pct,R2_cv_pct,Param_Combinations_Tried,Grid_Best_CV_RMSE,Grid_Worst_CV_RMSE,Grid_CV_RMSE_STD_Avg
0,dust,Random Forest,157.9989,14.6945,109.9174,10.6564,12.9115,1.4111,0.4151,0.1533,"{""model__max_depth"": null, ""model__min_samples...",1.0,1.0,9.3004,36.9238,8,143.1107,144.6904,18.1441
1,dust,XGBoost,158.4356,23.7198,111.5719,14.8468,13.2380,2.1119,0.4043,0.2135,"{""model__learning_rate"": 0.05, ""model__max_dep...",2.0,2.0,14.9713,52.8221,8,144.2619,148.7414,16.2737
2,dust,LightGBM,161.1379,20.8432,115.5919,16.5197,13.6848,2.1962,0.3913,0.1763,"{""model__learning_rate"": 0.03, ""model__n_estim...",3.0,3.0,12.9350,45.0588,8,152.4689,155.5354,18.3618
3,dust,Gradient Boosting,170.1659,29.1139,120.5475,17.4954,14.2160,2.1290,0.2981,0.3346,"{""model__learning_rate"": 0.03, ""model__max_dep...",4.0,4.0,17.1091,112.2515,8,151.2015,161.7260,16.7668
4,high_grown,Random Forest,225.3322,28.7180,156.9024,23.1968,19.0154,3.3432,0.2839,0.0847,"{""model__max_depth"": 10, ""model__min_samples_s...",1.0,1.0,12.7447,29.8312,8,226.4311,228.9641,39.4698
5,high_grown,LightGBM,227.7371,33.3049,161.4617,25.9648,19.6037,3.7562,0.2737,0.0459,"{""model__learning_rate"": 0.03, ""model__n_estim...",2.0,2.0,14.6243,16.7604,8,230.6975,239.5302,36.7314
6,high_grown,Gradient Boosting,232.1093,33.0956,162.8220,29.7617,19.8924,4.4911,0.2442,0.0692,"{""model__learning_rate"": 0.03, ""model__max_dep...",3.0,3.0,14.2586,28.3441,8,235.5007,246.7244,42.8957
7,high_grown,XGBoost,233.4988,31.8901,161.4631,26.6298,19.7608,3.8562,0.2340,0.0721,"{""model__learning_rate"": 0.03, ""model__max_dep...",4.0,4.0,13.6575,30.7964,8,231.7818,237.7434,40.1214
8,low_grown,Gradient Boosting,278.6583,63.6215,151.6967,19.9301,8.8090,0.9629,0.8599,0.0661,"{""model__learning_rate"": 0.05, ""model__max_dep...",1.0,1.0,22.8314,7.6873,8,291.9311,381.6667,29.8911
9,low_grown,Random Forest,285.7480,81.3579,155.7914,32.4983,9.0175,1.6040,0.8469,0.0913,"{""model__max_depth"": null, ""model__min_samples...",2.0,2.0,28.4719,10.7826,8,290.7005,295.3040,44.0410



Combined RMSE/R2 performance table:


,Segment,Model,RMSE_mean,R2_mean,RMSE_rank_in_segment,R2_rank_in_segment,RMSE_cv_pct,R2_cv_pct,Param_Combinations_Tried,Grid_Best_CV_RMSE,Grid_Worst_CV_RMSE,Grid_CV_RMSE_STD_Avg
0,dust,Random Forest,157.9989,0.4151,1.0,1.0,9.3004,36.9238,8,143.1107,144.6904,18.1441
1,dust,XGBoost,158.4356,0.4043,2.0,2.0,14.9713,52.8221,8,144.2619,148.7414,16.2737
2,dust,LightGBM,161.1379,0.3913,3.0,3.0,12.9350,45.0588,8,152.4689,155.5354,18.3618
3,dust,Gradient Boosting,170.1659,0.2981,4.0,4.0,17.1091,112.2515,8,151.2015,161.7260,16.7668
4,high_grown,Random Forest,225.3322,0.2839,1.0,1.0,12.7447,29.8312,8,226.4311,228.9641,39.4698
5,high_grown,LightGBM,227.7371,0.2737,2.0,2.0,14.6243,16.7604,8,230.6975,239.5302,36.7314
6,high_grown,Gradient Boosting,232.1093,0.2442,3.0,3.0,14.2586,28.3441,8,235.5007,246.7244,42.8957
7,high_grown,XGBoost,233.4988,0.2340,4.0,4.0,13.6575,30.7964,8,231.7818,237.7434,40.1214
8,low_grown,Gradient Boosting,278.6583,0.8599,1.0,1.0,22.8314,7.6873,8,291.9311,381.6667,29.8911
9,low_grown,Random Forest,285.7480,0.8469,2.0,2.0,28.4719,10.7826,8,290.7005,295.3040,44.0410



Saved:
- ../reports/tables/segmentwise_all_findings_table.csv
- ../reports/tables/segmentwise_rmse_r2_combined.csv
